# Burst Scheduler Playground

This notebook turns the theory repo into an executable sketch for burst scheduling. It models a single logical agent, a local execution anchor, and several cloud providers with different cost, speed, and data-transfer tradeoffs.

The goal is not to predict exact production behavior. The goal is to make the control-plane assumptions concrete enough to inspect, tune, and challenge.

In [ ]:
from dataclasses import dataclass, asdict
from pprint import pprint

@dataclass(frozen=True)
class Provider:
    name: str
    hourly_cost: float
    speed_factor: float
    queue_minutes: float
    transfer_penalty: float
    reliability: float
    gpu_ready: bool

@dataclass(frozen=True)
class Workload:
    name: str
    base_minutes_local: float
    requires_gpu: bool
    working_set_gb: float
    max_budget_usd: float
    min_reliability: float = 0.95

def estimate_minutes(provider: Provider, workload: Workload) -> float:
    compute_minutes = workload.base_minutes_local / provider.speed_factor
    return compute_minutes + provider.queue_minutes + provider.transfer_penalty * workload.working_set_gb

def estimate_cost(provider: Provider, workload: Workload) -> float:
    return estimate_minutes(provider, workload) / 60 * provider.hourly_cost

def eligible(provider: Provider, workload: Workload) -> bool:
    if workload.requires_gpu and not provider.gpu_ready:
        return False
    if provider.reliability < workload.min_reliability:
        return False
    return estimate_cost(provider, workload) <= workload.max_budget_usd

def score(provider: Provider, workload: Workload) -> float:
    minutes = estimate_minutes(provider, workload)
    cost = estimate_cost(provider, workload)
    return (
        30 * provider.speed_factor
        - 1.2 * minutes
        - 18 * cost
        - 10 * max(0.0, workload.min_reliability - provider.reliability)
    )

def evaluate(workload: Workload, providers: list[Provider]) -> list[dict]:
    rows = []
    for provider in providers:
        rows.append({
            "provider": provider.name,
            "eligible": eligible(provider, workload),
            "minutes": round(estimate_minutes(provider, workload), 1),
            "cost_usd": round(estimate_cost(provider, workload), 2),
            "score": round(score(provider, workload), 2),
            "gpu_ready": provider.gpu_ready,
            "reliability": provider.reliability,
        })
    return sorted(rows, key=lambda row: (not row["eligible"], -row["score"]))

In [ ]:
providers = [
    Provider("local-anchor", hourly_cost=0.55, speed_factor=1.0, queue_minutes=0.0, transfer_penalty=0.0, reliability=0.995, gpu_ready=True),
    Provider("cloud-a-cpu", hourly_cost=0.68, speed_factor=1.6, queue_minutes=1.0, transfer_penalty=0.25, reliability=0.985, gpu_ready=False),
    Provider("cloud-b-gpu", hourly_cost=2.10, speed_factor=4.8, queue_minutes=3.0, transfer_penalty=0.18, reliability=0.978, gpu_ready=True),
    Provider("cloud-c-spot", hourly_cost=0.92, speed_factor=2.8, queue_minutes=4.0, transfer_penalty=0.30, reliability=0.955, gpu_ready=True),
    Provider("cloud-d-low-latency", hourly_cost=1.25, speed_factor=2.1, queue_minutes=0.5, transfer_penalty=0.10, reliability=0.991, gpu_ready=True),
]

workloads = [
    Workload("repo-index-refresh", base_minutes_local=40, requires_gpu=False, working_set_gb=3.0, max_budget_usd=1.50),
    Workload("embedding-rebuild", base_minutes_local=180, requires_gpu=True, working_set_gb=12.0, max_budget_usd=8.00),
    Workload("artifact-dedup-pass", base_minutes_local=95, requires_gpu=False, working_set_gb=20.0, max_budget_usd=2.75),
]

for workload in workloads:
    print(f"\n=== {workload.name} ===")
    pprint(evaluate(workload, providers))

In [ ]:
target = workloads[1]
base_provider = next(provider for provider in providers if provider.name == "cloud-b-gpu")
print(f"Sensitivity for {target.name} on {base_provider.name}\n")

for penalty in [0.05, 0.10, 0.20, 0.35, 0.50]:
    adjusted = Provider(
        name=base_provider.name,
        hourly_cost=base_provider.hourly_cost,
        speed_factor=base_provider.speed_factor,
        queue_minutes=base_provider.queue_minutes,
        transfer_penalty=penalty,
        reliability=base_provider.reliability,
        gpu_ready=base_provider.gpu_ready,
    )
    minutes = estimate_minutes(adjusted, target)
    cost = estimate_cost(adjusted, target)
    print({
        "transfer_penalty": penalty,
        "minutes": round(minutes, 1),
        "cost_usd": round(cost, 2),
        "eligible": eligible(adjusted, target),
        "score": round(score(adjusted, target), 2),
    })

In [ ]:
def choose_provider(workload: Workload, providers: list[Provider]) -> dict:
    ranked = evaluate(workload, providers)
    for row in ranked:
        if row["eligible"]:
            return row
    return ranked[0]

decisions = {workload.name: choose_provider(workload, providers) for workload in workloads}
pprint(decisions)

## Takeaway

This notebook makes one design point visible: bursting is rarely just about raw speed. The winning provider changes when queue time, transfer weight, reliability floors, and budget ceilings move. That is the kind of scheduler behavior the repo is trying to theorize, not a simplistic `always send to the biggest GPU` policy.